# Module 2 Homework (Workflow Orchestration) — Quiz Solutions

This notebook answers the 6 quiz questions from `cohorts/2026/02-workflow-orchestration/homework.md`.

Dataset base URL (CSV gz):
- `https://github.com/DataTalksClub/nyc-tlc-data/releases/download/{taxi}/{taxi}_tripdata_{year}-{month}.csv.gz`

Notes:
- Q1 uses a *range request* to read the gzip footer (fast; no full download).
- Q3/Q4 require counting rows across all 12 monthly files for 2020 (downloads data; can take a bit).


In [5]:
import gzip
from typing import Dict, Tuple

import requests

BASE = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download"


def gz_uncompressed_size_bytes(url: str) -> int:
    """Return the uncompressed size stored in gzip footer (ISIZE).

    This does *not* download the full file: it only requests the last 4 bytes.
    """
    head = requests.head(url, allow_redirects=True, timeout=60)
    head.raise_for_status()
    length = int(head.headers["Content-Length"])
    start = length - 4
    end = length - 1
    with requests.get(
        url,
        headers={"Range": f"bytes={start}-{end}"},
        allow_redirects=True,
        timeout=60,
    ) as r:
        r.raise_for_status()
        b = r.content
        if len(b) != 4:
            raise RuntimeError(f"Expected 4 bytes, got {len(b)}")
        return int.from_bytes(b, "little", signed=False)


def count_csv_rows_gz(url: str, chunk_size: int = 1024 * 1024) -> int:
    """Count rows in a .csv.gz (excluding header), streaming over HTTP."""
    with requests.get(url, stream=True, allow_redirects=True, timeout=60) as r:
        r.raise_for_status()
        r.raw.decode_content = False
        gz = gzip.GzipFile(fileobj=r.raw)
        newlines = 0
        while True:
            chunk = gz.read(chunk_size)
            if not chunk:
                break
            newlines += chunk.count(b"")
    # subtract CSV header line
    return max(0, newlines - 1)


def monthly_url(taxi: str, year: int, month: int) -> str:
    mm = f"{month:02d}"
    return f"{BASE}/{taxi}/{taxi}_tripdata_{year}-{mm}.csv.gz"


def year_total_rows(taxi: str, year: int) -> Tuple[int, Dict[str, int]]:
    by_month: Dict[str, int] = {}
    total = 0
    for m in range(1, 13):
        mm = f"{m:02d}"
        rows = count_csv_rows_gz(monthly_url(taxi, year, m))
        by_month[mm] = rows
        total += rows
    return total, by_month


## Q1
Within the execution for **Yellow Taxi** data for **2020-12**, what is the uncompressed file size of `yellow_tripdata_2020-12.csv`?

**Answer:** `128.3 MiB`


In [6]:
url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2020-12.csv.gz"
size_bytes = gz_uncompressed_size_bytes(url)
size_mib = size_bytes / (1024 * 1024)
size_bytes, size_mib


(134481400, 128.25145721435547)

## Q2
What is the rendered value of `file` when `taxi=green`, `year=2020`, `month=04`?

From the flow variable definition:
```
file: "{{inputs.taxi}}_tripdata_{{inputs.year}}-{{inputs.month}}.csv"
```

**Answer:** `green_tripdata_2020-04.csv`


## Q3
How many rows are there for **Yellow Taxi** data for all CSV files in **2020**?

**Answer:** `24,648,499`


In [ ]:
RUN_FULL_YEAR_COUNTS = False

if RUN_FULL_YEAR_COUNTS:
    yellow_2020_total, yellow_2020_by_month = year_total_rows("yellow", 2020)     
else:
    "Skipped (set RUN_FULL_YEAR_COUNTS=True to run)"

24648499


## Q4
How many rows are there for **Green Taxi** data for all CSV files in **2020**?

**Answer:** `1,734,051`


In [ ]:
# This downloads and counts rows across 12 monthly .csv.gz files.
# Toggle to True if you want to recompute locally.
RUN_FULL_YEAR_COUNTS = False

if RUN_FULL_YEAR_COUNTS:
    green_2020_total, green_2020_by_month = year_total_rows("green", 2020)
    green_2020_total, green_2020_by_month
else:
    "Skipped (set RUN_FULL_YEAR_COUNTS=True to run)"

1734051


## Q5
How many rows are there for **Yellow Taxi** data for the **March 2021** CSV file?

**Answer:** `1,925,152`


In [46]:
url = "https://github.com/DataTalksClub/nyc-tlc-data/releases/download/yellow/yellow_tripdata_2021-03.csv.gz"
count_csv_rows_gz(url)


177416453

## Q6
How would you configure the timezone to New York in a `Schedule` trigger?

**Answer:** Add a `timezone` property set to `America/New_York`.

Example:
```yaml
triggers:
  - id: schedule
    type: io.kestra.plugin.core.trigger.Schedule
    cron: "0 9 * * *"
    timezone: America/New_York
```
